In [44]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', 1000)

# Fresh load - untouched raw data
df = pd.read_csv('../data/raw/india_job_market_2025.csv')

print(f"Starting shape: {df.shape}")
# Expected: (97929, 17)

Starting shape: (97929, 17)


In [45]:
# Identify rows missing all 5 critical columns simultaneously
critical_cols = ['minimumSalary', 'maximumSalary', 'minimumExperience', 'maximumExperience', 'tagsAndSkills']
systemic_failure_mask = df[critical_cols].isnull().all(axis=1)

print(f"Rows to drop (systemic failure): {systemic_failure_mask.sum()}")

df = df[~systemic_failure_mask].copy()

print(f"Shape after dropping systemic-failure rows: {df.shape}")
# Expected: (97358, 17)

Rows to drop (systemic failure): 571
Shape after dropping systemic-failure rows: (97358, 17)


In [46]:
full_dupes_count = df.duplicated().sum()
print(f"Full duplicate rows to drop: {full_dupes_count}")

df = df.drop_duplicates(keep='first').copy()

print(f"Shape after dropping full duplicates: {df.shape}")
# Expected: (97111, 17)

Full duplicate rows to drop: 247
Shape after dropping full duplicates: (97111, 17)


In [47]:
# Drop rows where BOTH jobId AND title match (true duplicates)
# This correctly spares the 11025020549 pair since their titles differ
true_dupe_mask = df.duplicated(subset=['jobId', 'title'], keep='first')

print(f"True partial duplicates to drop: {true_dupe_mask.sum()}")

df = df[~true_dupe_mask].copy()

print(f"Shape after resolving partial duplicates: {df.shape}")
# Expected: (97109, 17)

True partial duplicates to drop: 2
Shape after resolving partial duplicates: (97109, 17)


In [48]:
missing_company = df['companyName'].isnull().sum()
print(f"Rows missing companyName: {missing_company}")

df = df.dropna(subset=['companyName']).copy()

print(f"Shape after dropping missing companyName: {df.shape}")
# Expected: (97105, 17)

Rows missing companyName: 4
Shape after dropping missing companyName: (97105, 17)


In [49]:
print("=" * 60)
print("PHASE 5 - CHECKPOINT 1 SUMMARY")
print("=" * 60)
print(f"Original rows:        97,929")
print(f"Current rows:          {len(df):,}")
print(f"Total removed:         {97929 - len(df):,}")
print(f"Removal percentage:    {(97929 - len(df)) / 97929 * 100:.2f}%")
print(f"\nRemaining null check:")
print(df[critical_cols].isnull().sum())

PHASE 5 - CHECKPOINT 1 SUMMARY
Original rows:        97,929
Current rows:          97,105
Total removed:         824
Removal percentage:    0.84%

Remaining null check:
minimumSalary        0
maximumSalary        0
minimumExperience    0
maximumExperience    0
tagsAndSkills        0
dtype: int64


In [50]:
import re

def extract_work_mode(location_str):
    """
    Extracts work mode (Remote/Hybrid/Onsite) from the location string.
    Returns the work mode and does NOT modify the original string.
    """
    location_lower = str(location_str).lower()
    
    if 'remote' in location_lower:
        return 'Remote'
    elif 'hybrid' in location_lower:
        return 'Hybrid'
    else:
        return 'Onsite'

df['work_mode'] = df['location'].apply(extract_work_mode)

print(df['work_mode'].value_counts())
print(f"\nTotal: {df['work_mode'].value_counts().sum()}")

work_mode
Onsite    89452
Hybrid     5733
Remote     1920
Name: count, dtype: int64

Total: 97105


In [51]:
# Rows that were originally exactly "Remote" but did NOT get classified as Remote now
exactly_remote = df[df['location'].str.strip().str.lower() == 'remote']
print(f"Rows where location is exactly 'Remote': {len(exactly_remote)}")

not_classified_remote = exactly_remote[exactly_remote['work_mode'] != 'Remote']
print(f"Of those, how many were NOT classified as Remote: {len(not_classified_remote)}")
print(not_classified_remote[['location', 'work_mode']].head())

Rows where location is exactly 'Remote': 1914
Of those, how many were NOT classified as Remote: 0
Empty DataFrame
Columns: [location, work_mode]
Index: []


In [52]:
def extract_primary_city(location_str):
    """
    Extracts the primary (first-listed) city from a messy location string.
    Order of operations matters: strip work-mode prefix -> strip parentheses -> split on comma -> trim.
    """
    text = str(location_str)
    
    # Step 1: Remove work-mode prefixes like "Hybrid - " or "Remote - "
    text = re.sub(r'^(Hybrid|Remote)\s*-\s*', '', text, flags=re.IGNORECASE)
    
    # Step 2: Remove anything inside parentheses, e.g. "(Motera)" or "(Hennur +2)"
    text = re.sub(r'\(.*?\)', '', text)
    
    # Step 3: Split on comma, take the first part (primary city)
    primary = text.split(',')[0]
    
    # Step 4: Trim whitespace
    primary = primary.strip()
    
    return primary

df['primary_city'] = df['location'].apply(extract_primary_city)

print(f"Unique cities BEFORE standardization: {df['primary_city'].nunique()}")
print(f"\nTop 20 cities:")
print(df['primary_city'].value_counts().head(20))

Unique cities BEFORE standardization: 1305

Top 20 cities:
primary_city
Bengaluru          19292
Hyderabad          12308
Pune                9337
Mumbai              6591
Chennai             6398
Gurugram            5187
Noida               4930
Ahmedabad           2408
Kolkata             1956
Remote              1914
Navi Mumbai         1696
Thane               1027
Kochi                976
Jaipur               960
Mumbai Suburban      781
Coimbatore           746
Chandigarh           621
Mohali               615
Vadodara             588
Nagpur               576
Name: count, dtype: int64


In [53]:
# Check specifically for known spelling variants
known_variants = ['Bangalore', 'Bengaluru', 'Gurgaon', 'Gurugram', 'Cochin', 'Kochi', 'Pondicherry', 'Puducherry']
variant_check = df['primary_city'].value_counts()
for v in known_variants:
    if v in variant_check.index:
        print(f"{v:15s} : {variant_check[v]}")

Bengaluru       : 19292
Gurugram        : 5187
Kochi           : 976
Pondicherry     : 1
Puducherry      : 25


In [54]:
# See the bottom of the distribution - this reveals junk/rare entries
print(df['primary_city'].value_counts().tail(40))

primary_city
Chhipabarod     1
Garhshankar     1
Chitapur        1
Borio           1
Bijbehara       1
Indri           1
Lodhika         1
Azara           1
Seppa           1
Sattenapalle    1
Bathalapalle    1
Atreyapuram     1
Yellapur        1
Halvad          1
Dhemaji         1
Dongargaon      1
Kadegaon        1
Nawabganj       1
Manjhanpur      1
Janjgir         1
Tirodi          1
Singapore       1
Indora          1
Odisha          1
Moinabad        1
Berhampore      1
Baska           1
Muktsar         1
Nabadwip        1
Akbarpur        1
Beri            1
Hajipur         1
Islampur        1
Surajpur        1
Madhepura       1
Nevasa          1
Kawardha        1
Bokakhat        1
Bamanwas        1
Tumsar          1
Name: count, dtype: int64


In [55]:
# How many cities have fewer than 5 postings? (likely junk/noise/rare typos)
rare_cities = df['primary_city'].value_counts()
print(f"Cities with fewer than 5 postings: {(rare_cities < 5).sum()}")
print(f"Total rows affected by these rare cities: {rare_cities[rare_cities < 5].sum()}")

Cities with fewer than 5 postings: 839
Total rows affected by these rare cities: 1439


In [56]:
# --- 1. Fix spelling variant ---
df['primary_city'] = df['primary_city'].replace({'Pondicherry': 'Puducherry'})

# --- 2. Fix "Remote" leaking into city field ---
df.loc[df['primary_city'] == 'Remote', 'primary_city'] = 'Not Specified'

# --- 3. Flag the one non-India entry ---
df.loc[df['primary_city'] == 'Singapore', 'primary_city'] = 'International/Remote'

# --- 4. Flag the state-name-as-city anomaly ---
df.loc[df['primary_city'] == 'Odisha', 'primary_city'] = 'Unspecified'

# --- 5. Build city_tier_group for clean visual rollups ---
city_counts = df['primary_city'].value_counts()
rare_cities = city_counts[city_counts < 5].index

def assign_city_tier_group(city):
    if city in ['Not Specified', 'International/Remote', 'Unspecified']:
        return city
    elif city in rare_cities:
        return 'Other (Tier 3/4 Town)'
    else:
        return city

df['city_tier_group'] = df['primary_city'].apply(assign_city_tier_group)

# --- 6. Build metro_region rollup for major metros ---
metro_map = {
    'Mumbai': 'Mumbai Metro', 'Navi Mumbai': 'Mumbai Metro', 
    'Thane': 'Mumbai Metro', 'Mumbai Suburban': 'Mumbai Metro',
    'Gurugram': 'Delhi NCR', 'Noida': 'Delhi NCR', 
    'New Delhi': 'Delhi NCR', 'Delhi': 'Delhi NCR', 'Faridabad': 'Delhi NCR', 'Ghaziabad': 'Delhi NCR'
}

df['metro_region'] = df['primary_city'].map(metro_map).fillna(df['primary_city'])

# --- Verification ---
print(f"Unique primary_city values: {df['primary_city'].nunique()}")
print(f"Unique city_tier_group values: {df['city_tier_group'].nunique()}")
print(f"\nTop 15 city_tier_group:")
print(df['city_tier_group'].value_counts().head(15))
print(f"\nTop 10 metro_region:")
print(df['metro_region'].value_counts().head(10))

Unique primary_city values: 1304
Unique city_tier_group values: 469

Top 15 city_tier_group:
city_tier_group
Bengaluru                19292
Hyderabad                12308
Pune                      9337
Mumbai                    6591
Chennai                   6398
Gurugram                  5187
Noida                     4930
Ahmedabad                 2408
Kolkata                   1956
Not Specified             1914
Navi Mumbai               1696
Other (Tier 3/4 Town)     1436
Thane                     1027
Kochi                      976
Jaipur                     960
Name: count, dtype: int64

Top 10 metro_region:
metro_region
Bengaluru        19292
Hyderabad        12308
Delhi NCR        11283
Mumbai Metro     10095
Pune              9337
Chennai           6398
Ahmedabad         2408
Kolkata           1956
Not Specified     1914
Kochi              976
Name: count, dtype: int64


In [57]:
# Confirm the math
total_unique = df['primary_city'].nunique()
rare_count = (df['primary_city'].value_counts() < 5).sum()
non_rare_count = total_unique - rare_count

print(f"Total unique cities: {total_unique}")
print(f"Rare cities (<5 postings): {rare_count}")
print(f"Non-rare cities (kept as-is): {non_rare_count}")
print(f"Expected city_tier_group count: {non_rare_count} + 1 (Other bucket) + flags already inside non_rare or separate")

Total unique cities: 1304
Rare cities (<5 postings): 838
Non-rare cities (kept as-is): 466
Expected city_tier_group count: 466 + 1 (Other bucket) + flags already inside non_rare or separate


In [58]:
# Better strategy for a DASHBOARD-ready grouping: Top 25 cities explicitly, everything else -> Other
top_25_cities = df['primary_city'].value_counts().head(25).index.tolist()

def assign_city_tier_group_v2(city):
    if city in ['Not Specified', 'International/Remote', 'Unspecified']:
        return city
    elif city in top_25_cities:
        return city
    else:
        return 'Other Cities'

df['city_tier_group'] = df['primary_city'].apply(assign_city_tier_group_v2)

print(f"Unique city_tier_group values: {df['city_tier_group'].nunique()}")
print(f"\nFull distribution:")
print(df['city_tier_group'].value_counts())

Unique city_tier_group values: 28

Full distribution:
city_tier_group
Bengaluru               19292
Other Cities            15936
Hyderabad               12308
Pune                     9337
Mumbai                   6591
Chennai                  6398
Gurugram                 5187
Noida                    4930
Ahmedabad                2408
Kolkata                  1956
Not Specified            1914
Navi Mumbai              1696
Thane                    1027
Kochi                     976
Jaipur                    960
Mumbai Suburban           781
Coimbatore                746
Chandigarh                621
Mohali                    615
Vadodara                  588
Nagpur                    576
Surat                     499
Lucknow                   473
Nashik                    460
New Delhi                 454
Faridabad                 374
International/Remote        1
Unspecified                 1
Name: count, dtype: int64


In [59]:
print(df['title'].value_counts().head(30))

title
Application Developer             2033
Application Lead                  1346
Sales Executive                    576
Software Development Engineer      497
Business Development Executive     455
Trust & Safety New Associate       348
Data Engineer                      338
Business Development Manager       323
Sales Manager                      319
Java Full Stack Developer          282
Java Developer                     253
Relationship Manager               232
Area Sales Manager                 217
Application Support Engineer       217
Accountant                         215
Customer Support Executive         213
Senior Software Engineer           206
Account Executive                  205
Graphic Designer                   188
Software Development Lead          181
Business Development Associate     177
Business Analyst                   172
Technical Lead - L1                165
Software Engineer                  164
AI / ML Engineer                   153
Sales Officer Home 

In [60]:
# Are there any obvious non-tech roles still in here that we flagged back in Phase 4?
non_tech_keywords = ['Sales', 'HR', 'Recruiter', 'Marketing', 'Accountant', 'Teacher', 'Nurse', 'Driver']
for kw in non_tech_keywords:
    count = df['title'].str.contains(kw, case=False, na=False).sum()
    print(f"{kw:12s}: {count}")

Sales       : 9181
HR          : 1613
Recruiter   : 935
Marketing   : 1745
Accountant  : 748
Teacher     : 454
Nurse       : 189
Driver      : 70


In [61]:
import re

# Keywords that need WORD BOUNDARY matching (short/ambiguous words that cause false matches as substrings)
# Keywords NOT in this list will still use substring matching (safe for longer, specific phrases)

tech_keywords = [
    'Developer', 'Engineer', 'Programmer', 'Architect', 'DevOps',
    'Data Scientist', 'Data Analyst', 'Data Engineer', 'Analytics',
    'Software', 'Application', 'Full Stack', 'Fullstack', 'Frontend', 'Backend',
    'Java', 'Python', 'SQL', 'Cloud', 'AWS', 'Azure', r'\bAI\b', r'\bML\b',
    'Machine Learning', r'\bQA\b', 'Test Engineer', 'SDET', 'Network Engineer',
    'System Administrator', 'Database Administrator', r'\bDBA\b', 'Cybersecurity',
    'Security Engineer', 'Security Architect', 'Business Analyst', 'Business Architect',
    r'\bIT\b', 'Technical Lead', 'Technical Project Manager', 'Technical Architect',
    'Web Developer', 'Mobile Developer', 'UI/UX', 'Solution Architect',
    'Automation', 'Embedded', 'Firmware', 'Blockchain', 'Quality Analyst',
    'Quality Engineer'
]

exclude_keywords = [
    r'\bSales\b',  # word boundary - won't match "Salesforce"
    'Business Development', 'Marketing', r'\bHR\b', 'Human Resource',
    'Recruiter', 'Recruitment', 'Accountant', 'Accounting', 'Finance Officer',
    'Financial Analyst', 'Teacher', 'Professor', 'Nurse', 'Doctor', 'Driver',
    'Customer Support', 'Customer Care', 'Relationship Manager', 'Account Executive',
    'Field Sales', 'Telecaller', 'Insurance', 'Loan', 'Banking Officer',
    'Operations Executive', 'Logistics', 'Warehouse', 'Retail', 'Store Manager',
    'Hospitality', 'Chef', 'Housekeeping', 'Security Guard',
    'Record To Report', 'Order To Cash', 'Procure To Pay', 'Hire To Retire',
    'Technical Recruiter', 'Sales Engineer'
]

def is_tech_role_v2(title):
    title_str = str(title)
    
    # Check excludes first (using regex for word-boundary terms, substring for phrases)
    for kw in exclude_keywords:
        if kw.startswith(r'\b'):
            if re.search(kw, title_str, re.IGNORECASE):
                return False
        else:
            if kw.lower() in title_str.lower():
                return False
    
    # Check includes
    for kw in tech_keywords:
        if kw.startswith(r'\b'):
            if re.search(kw, title_str, re.IGNORECASE):
                return True
        else:
            if kw.lower() in title_str.lower():
                return True
    
    return False

df['is_tech_role'] = df['title'].apply(is_tech_role_v2)

print(f"Tech roles identified: {df['is_tech_role'].sum():,}")
print(f"Non-tech roles identified: {(~df['is_tech_role']).sum():,}")
print(f"Tech role percentage: {df['is_tech_role'].mean()*100:.2f}%")

Tech roles identified: 32,283
Non-tech roles identified: 64,822
Tech role percentage: 33.25%


In [62]:
# See what got classified as tech - spot check the top 30
print("TOP 30 TECH-CLASSIFIED TITLES:")
print(df[df['is_tech_role']]['title'].value_counts().head(30))

TOP 30 TECH-CLASSIFIED TITLES:
title
Application Developer                    2033
Application Lead                         1346
Software Development Engineer             497
Data Engineer                             338
Java Full Stack Developer                 282
Java Developer                            253
Application Support Engineer              217
Senior Software Engineer                  206
Software Development Lead                 181
Business Analyst                          172
Technical Lead - L1                       165
Software Engineer                         164
AI / ML Engineer                          153
Developer - L3                            142
Application Designer                      118
Senior Data Engineer                      116
Application Tech Support Practitioner     115
Full Stack Engineer                       112
Data Scientist                            108
Security Architect                        102
Solution Architect                        1

In [63]:
# See what got classified as non-tech but CONTAINS a tech-sounding word - catch false negatives
tech_sounding_but_excluded = df[~df['is_tech_role']]['title'].str.contains(
    'Engineer|Developer|Analyst|Technical', case=False, na=False
)
print(f"\nPotential false negatives (tech-sounding but excluded): {tech_sounding_but_excluded.sum()}")
print(df[~df['is_tech_role']][tech_sounding_but_excluded]['title'].value_counts().head(20))


Potential false negatives (tech-sounding but excluded): 4150
title
Analyst                                        84
Financial Analyst                              68
Sales Engineer                                 66
Record To Report Ops Analyst                   35
Senior Analyst                                 33
Delivery Operations Senior Analyst             33
Analyst - KYC                                  33
Sales Executive / Graduate Engineer Trainee    28
Technical Writer                               22
Technical Recruiter                            22
Business Interlock Analyst                     22
Record To Report Ops Senior Analyst            22
I&F Decision Sci Practitioner Sr Analyst       21
Order To Cash Operations Analyst               18
Business Interlock Senior Analyst              17
Trust & Safety Analyst                         15
Procure To Pay Operations Senior Analyst       14
Cyber Security Analyst - L4                    14
HR Service Delivery Analyst     

In [64]:
print(f"Tech roles identified: {df['is_tech_role'].sum():,}")
print(f"Tech role percentage: {df['is_tech_role'].mean()*100:.2f}%")

# Verify Salesforce Developer is now correctly captured
print("\nSalesforce Developer check:")
print(df[df['title'].str.contains('Salesforce', case=False, na=False)]['is_tech_role'].value_counts())

# Verify Technical Project Manager is now correctly captured
print("\nTechnical Project Manager check:")
print(df[df['title'].str.contains('Technical Project Manager', case=False, na=False)]['is_tech_role'].value_counts())

# Re-run the false negative check to see what's left
tech_sounding_but_excluded = df[~df['is_tech_role']]['title'].str.contains(
    'Engineer|Developer|Analyst|Technical', case=False, na=False
)
print(f"\nRemaining potential false negatives: {tech_sounding_but_excluded.sum()}")
print(df[~df['is_tech_role']][tech_sounding_but_excluded]['title'].value_counts().head(15))

Tech roles identified: 32,283
Tech role percentage: 33.25%

Salesforce Developer check:
is_tech_role
True     237
False    119
Name: count, dtype: int64

Technical Project Manager check:
is_tech_role
True    69
Name: count, dtype: int64

Remaining potential false negatives: 4150
title
Analyst                                        84
Financial Analyst                              68
Sales Engineer                                 66
Record To Report Ops Analyst                   35
Senior Analyst                                 33
Delivery Operations Senior Analyst             33
Analyst - KYC                                  33
Sales Executive / Graduate Engineer Trainee    28
Technical Writer                               22
Technical Recruiter                            22
Business Interlock Analyst                     22
Record To Report Ops Senior Analyst            22
I&F Decision Sci Practitioner Sr Analyst       21
Order To Cash Operations Analyst               18
Business Inter

In [65]:
print(df[df['title'].str.contains('Salesforce', case=False, na=False) & (~df['is_tech_role'])]['title'].value_counts().head(15))

title
Salesforce Administrator                                   9
Salesforce Marketing Cloud Developer                       5
Package Consultant-Salesforce                              5
Salesforce Lead                                            4
Senior Salesforce Consultant                               3
Salesforce Consultant                                      3
Salesforce Nonprofit, Sr Associate                         2
Salesforce Leader                                          2
Salesforce Marketing Cloud Professional                    2
Consultant, Salesforce development                         2
AD/SM Program Project Management Salesforce                2
Salesforce lead                                            1
Salesforce CPQ Administrator                               1
Salesforce Vlocity/Omnistudio | PAN India | 6-10 yrs !!    1
Senior Salesforce Administrator                            1
Name: count, dtype: int64


In [66]:
# Add missing tech-specific terms
tech_keywords_v3 = tech_keywords + [
    'Cyber Security', 'Security Analyst', 'Cloud Security', 
    'Information Security', 'SOC Analyst', 'Penetration Test'
]

def is_tech_role_v3(title):
    title_str = str(title)
    
    for kw in exclude_keywords:
        if kw.startswith(r'\b'):
            if re.search(kw, title_str, re.IGNORECASE):
                return False
        else:
            if kw.lower() in title_str.lower():
                return False
    
    for kw in tech_keywords_v3:
        if kw.startswith(r'\b'):
            if re.search(kw, title_str, re.IGNORECASE):
                return True
        else:
            if kw.lower() in title_str.lower():
                return True
    
    return False

df['is_tech_role'] = df['title'].apply(is_tech_role_v3)

print(f"Tech roles identified: {df['is_tech_role'].sum():,}")
print(f"Tech role percentage: {df['is_tech_role'].mean()*100:.2f}%")

# Verify Cyber Security Analyst now correctly captured
print("\nCyber Security Analyst check:")
print(df[df['title'].str.contains('Cyber Security Analyst', case=False, na=False)]['is_tech_role'].value_counts())

Tech roles identified: 32,438
Tech role percentage: 33.41%

Cyber Security Analyst check:
is_tech_role
True    27
Name: count, dtype: int64


In [67]:
# Create the tech-only dataset
df_tech = df[df['is_tech_role'] == True].copy()

print(f"Full cleaned dataset: {len(df):,} rows")
print(f"Tech-only dataset: {len(df_tech):,} rows")
print(f"Tech-only percentage: {len(df_tech)/len(df)*100:.2f}%")

# Reset index for cleanliness going forward
df_tech = df_tech.reset_index(drop=True)

print(f"\nFinal df_tech shape: {df_tech.shape}")

Full cleaned dataset: 97,105 rows
Tech-only dataset: 32,438 rows
Tech-only percentage: 33.41%

Final df_tech shape: (32438, 22)


In [68]:
print(df_tech.columns.tolist())

['title', 'jobId', 'currency', 'jobUploaded', 'companyName', 'tagsAndSkills', 'experience', 'salary', 'location', 'companyId', 'ReviewsCount', 'AggregateRating', 'jobDescription', 'minimumSalary', 'maximumSalary', 'minimumExperience', 'maximumExperience', 'work_mode', 'primary_city', 'city_tier_group', 'metro_region', 'is_tech_role']


In [69]:
def assign_role_category_v3(title):
    t = str(title).lower()
    
    # ML / AI first
    if any(kw in t for kw in ['machine learning', 'ml engineer', 'ai engineer',
                               'ai/ml', 'ai / ml', 'nlp', 'computer vision',
                               'llm', 'large language model', 'generative ai',
                               'deep learning', 'rpa']):
        return 'ML / AI Engineer'
    
    # AI Trainer — new 2025 role type (check before generic AI catch)
    if any(kw in t for kw in ['ai trainer', 'ai tutor', 'ai content writer',
                               'data labeler', 'data annotator', 'rlhf']):
        return 'AI Trainer / Data Labeler'
    
    # Data Science
    if any(kw in t for kw in ['data scientist', 'data science']):
        return 'Data Scientist'
    
    # Data Engineering
    if any(kw in t for kw in ['data engineer', 'etl', 'big data',
                               'data pipeline', 'spark', 'hadoop', 'databricks']):
        return 'Data Engineer'
    
    # Data Analyst / BI — added adobe analytics
    if any(kw in t for kw in ['data analyst', 'business intelligence',
                               'bi analyst', 'bi developer', 'analytics engineer',
                               'reporting analyst', 'adobe analytics']):
        return 'Data Analyst'
    
    # Security
    if any(kw in t for kw in ['security', 'cybersecurity', 'cyber security',
                               'soc analyst', 'penetration', 'vapt', 'infosec']):
        return 'Security Engineer'
    
    # DevOps / Cloud — added azure administrator, cloud admin
    if any(kw in t for kw in ['devops', 'cloud engineer', 'sre', 'site reliability',
                               'platform engineer', 'infrastructure engineer',
                               'aws engineer', 'azure engineer', 'kubernetes',
                               'docker engineer', 'azure administrator',
                               'cloud administrator', 'aws administrator']):
        return 'DevOps / Cloud Engineer'
    
    # QA / Testing — added 'tester', 'app automation'
    if any(kw in t for kw in ['quality engineer', 'quality analyst', 'qa ', 'qe ',
                               'test engineer', 'sdet', 'automation tester',
                               'manual tester', 'testing', 'test automation',
                               'test lead', 'automation lead', 'tester',
                               'automation eng']):
        return 'QA / Test Engineer'
    
    # Frontend
    if any(kw in t for kw in ['frontend', 'front end', 'front-end', 'ui developer',
                               'react developer', 'angular developer',
                               'vue developer', 'ui engineer', 'ui/ux',
                               'ux designer', 'ui designer']):
        return 'Frontend Developer'
    
    # Mobile
    if any(kw in t for kw in ['android', 'ios developer', 'flutter',
                               'react native', 'mobile developer', 'mobile engineer']):
        return 'Mobile Developer'
    
    # Backend
    if any(kw in t for kw in ['backend', 'back end', 'back-end', 'node developer',
                               'django developer', 'spring boot', 'api developer']):
        return 'Backend Developer'
    
    # Full Stack
    if any(kw in t for kw in ['full stack', 'fullstack', 'full-stack',
                               'mern', 'mean stack', 'lamp stack']):
        return 'Full Stack Developer'
    
    # Technical Lead / Architect — added IT manager variants
    if any(kw in t for kw in ['technical lead', 'tech lead', 'solution architect',
                               'enterprise architect', 'technology architect',
                               'application architect', 'it architect',
                               'data architect', 'software architect',
                               'cloud architect', 'security architect',
                               'servicenow architect', 'technical architect',
                               'senior architect', 'junior architect',
                               'architect', 'development lead', 'software lead',
                               'team leader', 'team lead',
                               'technical project manager',
                               'technical program manager',
                               'software manager', 'development manager',
                               'engineering manager', 'it manager',
                               'it project manager', 'it project coordinator',
                               'it consulting', 'unit manager - it',
                               'manager - software']):
        return 'Technical Lead / Architect'
    
    # Database
    if any(kw in t for kw in ['dba', 'database administrator', 'sql server dba',
                               'oracle dba', 'database engineer', 'mysql dba']):
        return 'Database Administrator'
    
    # IT Support / Sysadmin — added specialist, ITSM, coordinator
    if any(kw in t for kw in ['system administrator', 'sysadmin', 'it support',
                               'network engineer', 'network administrator',
                               'helpdesk', 'help desk', 'desktop support',
                               'l1 support', 'l2 support', 'l3 support',
                               'tech support', 'it executive',
                               'support practitioner', 'application support',
                               'application specialist', 'it service management',
                               'service management representative']):
        return 'IT Support / Sysadmin'
    
    # Product Manager
    if any(kw in t for kw in ['product manager', 'product owner',
                               'program manager', 'technical program manager']):
        return 'Product Manager'
    
    # Business Analyst
    if any(kw in t for kw in ['business analyst', 'business architect',
                               'systems analyst', 'functional consultant',
                               'business systems', 'process analyst',
                               'business process']):
        return 'Business Analyst'
    
    # Generic Software Engineer — added 'engr' abbreviation
    if any(kw in t for kw in ['software engineer', 'software developer',
                               'application developer', 'application lead',
                               'application designer', 'developer', 'engineer',
                               'programmer', 'java', 'python developer',
                               'dot net', '.net', 'php', 'scala', 'golang',
                               'rust developer', 'c++ developer', 'embedded',
                               'engr ']):
        return 'Software Engineer'
    
    return 'Other Tech'

df_tech['role_category'] = df_tech['title'].apply(assign_role_category_v3)

print(df_tech['role_category'].value_counts())
print(f"\nTotal categories: {df_tech['role_category'].nunique()}")
print(f"Other Tech percentage: {(df_tech['role_category'] == 'Other Tech').mean()*100:.2f}%")

# See what's genuinely left
print("\nRemaining Other Tech titles:")
print(df_tech[df_tech['role_category'] == 'Other Tech']['title'].value_counts().head(20))

role_category
Software Engineer             16676
Technical Lead / Architect     2827
QA / Test Engineer             2035
Other Tech                     1711
Full Stack Developer           1700
Data Engineer                  1300
DevOps / Cloud Engineer        1083
IT Support / Sysadmin           839
ML / AI Engineer                702
Security Engineer               623
Backend Developer               559
Frontend Developer              496
Business Analyst                493
Data Analyst                    380
Mobile Developer                351
Data Scientist                  332
Database Administrator          197
AI Trainer / Data Labeler        89
Product Manager                  45
Name: count, dtype: int64

Total categories: 19
Other Tech percentage: 5.27%

Remaining Other Tech titles:
title
Package Consultant-Oracle Finance Cloud                                   16
Package Consultant-Oracle SCM Cloud                                        8
TechOps - DE - CloudOpsAMS - GWPoli

In [70]:
print(df_tech[df_tech['role_category'] == 'Other Tech']['title'].value_counts().head(25))

title
Package Consultant-Oracle Finance Cloud                                   16
Package Consultant-Oracle SCM Cloud                                        8
TechOps - DE - CloudOpsAMS - GWPolicy - Staff                              6
Oracle Cloud Benefits Professional                                         5
Package Consultant-Oracle Enterprise Performance Management Cloud          5
TechOps-DE-CloudOpsAMS-GWPolicy-Senior                                     5
Package Consultant-Oracle Cloud HCM                                        5
Hiring Certified HCC Coder/QA                                              4
IT Head                                                                    4
Senior Applications Support Analyst                                        4
Oracle Supply Chain And CRM-Ignition Application Administrator-Manager     4
Software Configuration Lead                                                4
Analytics and Modeling Senior Analyst                                 

In [71]:
print(df_tech[df_tech['title'].str.contains('Technical Project Manager', case=False, na=False)]['role_category'].value_counts())

role_category
Technical Lead / Architect    55
ML / AI Engineer              11
Security Engineer              1
Data Scientist                 1
Data Engineer                  1
Name: count, dtype: int64


In [72]:
# Quick debug - why is Software Configuration Lead not caught?
test_titles = ['IT Head', 'Software Configuration Lead', 
               'Analytics and Modeling Senior Analyst',
               'Software Designer', 'Manager, Software Development',
               'Software Development Specialist', 'AWS + Snowflake']
for t in test_titles:
    print(f"{t:45s} → {assign_role_category_v3(t)}")

IT Head                                       → Other Tech
Software Configuration Lead                   → Other Tech
Analytics and Modeling Senior Analyst         → Other Tech
Software Designer                             → Other Tech
Manager, Software Development                 → Other Tech
Software Development Specialist               → Other Tech
AWS + Snowflake                               → Other Tech


In [73]:
# Sample 50 random tagsAndSkills values to check delimiter consistency
sample = df_tech['tagsAndSkills'].dropna().sample(50, random_state=42).tolist()
for s in sample[:15]:
    print(repr(s))

'software development,software testing,sas viya,root cause analysis,software development life cycle,sas macros,sas,sas sql'
'Audio Visual,Crestron,Video Conferencing,AV,Audio Conferencing,Projector,Installation,Digital Signage'
'application lifecycle management,debugging,interface design,application development,design principles,css,web services,hibernate'
'Servicenow Development,Snow Developer,Snow Development,Snow,Itam,ITSM,CMDB,Hrsd'
'ncvt,python,natural language processing,mechanical engineering,production,grinding,preventive maintenance,assembling'
'Firebase,Restfull Api,restful,Core Data,Swift Ui,Graphql,VOIP,JSON'
'sap,supply chain,sap configuration,sap mm materials management,inventory management,3d modeling,sap project management,software testing'
'software testing,automation testing,security testing,testing methodologies,api testing,defect tracking,manual testing,stlc'
'networking,alerts,serverless architecture,encryption,access control,kubernetes,python,c++'
'Change manageme

In [74]:
# Step 8a: Clean and split tagsAndSkills into lists
df_tech['skills_list'] = (
    df_tech['tagsAndSkills']
    .fillna('')
    .apply(lambda x: [skill.strip().title() for skill in x.split(',') if skill.strip() != ''])
)

# Quick check - how many skills per posting on average?
skill_counts = df_tech['skills_list'].apply(len)
print(f"Average skills per posting: {skill_counts.mean():.2f}")
print(f"Median skills per posting:  {skill_counts.median():.0f}")
print(f"Max skills in one posting:  {skill_counts.max()}")
print(f"Min skills in one posting:  {skill_counts.min()}")
print(f"Postings with 0 skills:     {(skill_counts == 0).sum()}")

Average skills per posting: 7.83
Median skills per posting:  8
Max skills in one posting:  8
Min skills in one posting:  1
Postings with 0 skills:     0


In [75]:
# Step 8b: Create the exploded long-format skills table
df_skills = df_tech[['jobId', 'title', 'role_category', 'primary_city', 
                      'city_tier_group', 'metro_region', 'work_mode',
                      'minimumSalary', 'maximumSalary', 
                      'minimumExperience', 'maximumExperience',
                      'companyName', 'skills_list']].copy()

df_skills = df_skills.explode('skills_list').rename(columns={'skills_list': 'skill'})

# Drop empty skill rows (postings that had no tags)
df_skills = df_skills[df_skills['skill'] != ''].copy()
df_skills = df_skills.reset_index(drop=True)

print(f"Long-format skills table shape: {df_skills.shape}")
print(f"Total skill mentions: {len(df_skills):,}")
print(f"Unique skills (raw, before normalization): {df_skills['skill'].nunique():,}")
print(f"\nTop 30 raw skills:")
print(df_skills['skill'].value_counts().head(30))

Long-format skills table shape: (253831, 13)
Total skill mentions: 253,831
Unique skills (raw, before normalization): 18,740

Top 30 raw skills:
skill
Python                     4959
Java                       3734
Css                        3661
Sql                        3099
C#                         2696
Javascript                 2551
Development                2143
Sap                        2009
Project Management         1830
Rest                       1815
Aws                        1705
C++                        1556
Continuous Integration     1544
Agile                      1536
Automation                 1511
Application Development    1498
Troubleshooting            1494
Microservices              1484
Spring Boot                1463
Software Development       1421
Oracle                     1364
Kubernetes                 1333
Software Testing           1319
Html                       1276
Web Services               1272
React.Js                   1226
Networking       

In [76]:
df_skills[df_skills['skill'] == 'Python']['maximumSalary'].median()

np.float64(0.0)

In [77]:
# Normalization dictionary - fixes acronyms, variants, and noise
skill_normalization = {
    # Acronym casing fixes (broken by .title())
    'Css': 'CSS', 'Sql': 'SQL', 'Aws': 'AWS', 'Sap': 'SAP',
    'Html': 'HTML', 'Api': 'API', 'Apis': 'APIs', 'Sdk': 'SDK',
    'Ui': 'UI', 'Ux': 'UX', 'Etl': 'ETL', 'Crm': 'CRM',
    'Erp': 'ERP', 'Ci/Cd': 'CI/CD', 'Cicd': 'CI/CD',
    'Ci': 'CI', 'Cd': 'CD', 'Ml': 'ML', 'Ai': 'AI',
    'Nlp': 'NLP', 'Oop': 'OOP', 'Orm': 'ORM', 'Mvc': 'MVC',
    'Jira': 'JIRA', 'Git': 'Git', 'Gcp': 'GCP', 'Rpa': 'RPA',
    'Devops': 'DevOps', 'Salesforce': 'Salesforce',
    'Javascript': 'JavaScript', 'Typescript': 'TypeScript',
    'Mongodb': 'MongoDB', 'Mysql': 'MySQL', 'Postgresql': 'PostgreSQL',
    'Graphql': 'GraphQL', 'Github': 'GitHub', 'Gitlab': 'GitLab',
    'Ios': 'iOS', 'Macos': 'MacOS', 'Linkedin': 'LinkedIn',
    'React.Js': 'ReactJS', 'React Js': 'ReactJS', 'Reactjs': 'ReactJS',
    'Node.Js': 'Node.js', 'Node Js': 'Node.js', 'Nodejs': 'Node.js',
    'Vue.Js': 'Vue.js', 'Angular.Js': 'AngularJS',
    'Dot Net': '.NET', 'Dotnet': '.NET', '.Net': '.NET',
    'Rest': 'REST API', 'Restful': 'REST API', 'Rest Api': 'REST API',
    'Restfull Api': 'REST API',
    'Aws Lambda': 'AWS Lambda',
    'Sap Mm Materials Management': 'SAP MM',
    'Sap Sd': 'SAP SD', 'Sap Hana': 'SAP HANA',
    'Ms Excel': 'Microsoft Excel', 'Ms Word': 'Microsoft Word',
    'Power Bi': 'Power BI', 'Powerbi': 'Power BI',
    'Tableau': 'Tableau', 'Looker': 'Looker',
    'Spring Boot': 'Spring Boot',
    'C#': 'C#', 'C++': 'C++',
    'Continuous Integration': 'CI/CD',
    'Snow': 'Snowflake', 'Snow Developer': 'Snowflake',
    'Snow Development': 'Snowflake',
}

# Generic noise skills to remove entirely
noise_skills = {
    'Development', 'Application Development', 'Software Development',
    'Software Development Life Cycle', 'Automation', 'Programming',
    'Troubleshooting', 'Debugging', 'Testing', 'Management',
    'Communication', 'Teamwork', 'Leadership', 'Problem Solving',
    'Analytical Skills', 'Team Player', 'Time Management',
    'Interpersonal Skills', 'Attention To Detail', 'Work Ethic'
}

# Apply normalization
df_skills['skill'] = df_skills['skill'].replace(skill_normalization)

# Remove noise skills
df_skills = df_skills[~df_skills['skill'].isin(noise_skills)].copy()
df_skills = df_skills.reset_index(drop=True)

print(f"Skills after normalization: {len(df_skills):,}")
print(f"Unique skills after normalization: {df_skills['skill'].nunique():,}")
print(f"\nTop 30 normalized skills:")
print(df_skills['skill'].value_counts().head(30))

Skills after normalization: 242,998
Unique skills after normalization: 18,707

Top 30 normalized skills:
skill
Python                4959
Java                  3734
CSS                   3661
SQL                   3099
C#                    2696
CI/CD                 2562
JavaScript            2551
SAP                   2009
REST API              1981
Project Management    1830
AWS                   1705
C++                   1556
Agile                 1536
Microservices         1484
Spring Boot           1463
Oracle                1364
Kubernetes            1333
Software Testing      1319
ReactJS               1305
HTML                  1276
Web Services          1272
Networking            1033
Git                    968
Machine Learning       966
Hibernate              951
Microsoft Azure        946
DevOps                 912
Angular                906
Computer Science       866
Automation Testing     862
Name: count, dtype: int64


In [78]:
# Add to df_tech (main dataset)
df_tech['salary_midpoint'] = (df_tech['minimumSalary'] + df_tech['maximumSalary']) / 2

# Add to df_skills (long format - already has min/max)
df_skills['salary_midpoint'] = (df_skills['minimumSalary'] + df_skills['maximumSalary']) / 2

print(f"Salary midpoint stats:")
print(df_tech['salary_midpoint'].describe())

Salary midpoint stats:
count    3.243800e+04
mean     2.352740e+05
std      6.603592e+05
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      2.500000e+07
Name: salary_midpoint, dtype: float64


In [79]:
additional_noise = {'Computer Science', 'Information Technology', 
                    'B.Tech', 'Engineering', 'Graduate'}
df_skills = df_skills[~df_skills['skill'].isin(additional_noise)].copy()
df_skills = df_skills.reset_index(drop=True)
print(f"Skills after additional noise removal: {len(df_skills):,}")

Skills after additional noise removal: 241,109


In [80]:
# How many rows have zero salary?
zero_salary = (df_tech['salary_midpoint'] == 0).sum()
disclosed = (df_tech['salary_midpoint'] > 0).sum()

print(f"Rows with salary = 0 (not disclosed): {zero_salary:,}")
print(f"Rows with salary > 0 (disclosed):     {disclosed:,}")
print(f"Salary disclosure rate:                {disclosed/len(df_tech)*100:.2f}%")

# What does the salary column say for these zero rows?
print(f"\nSalary string for zero-midpoint rows:")
print(df_tech[df_tech['salary_midpoint'] == 0]['salary'].value_counts().head(10))

Rows with salary = 0 (not disclosed): 26,062
Rows with salary > 0 (disclosed):     6,376
Salary disclosure rate:                19.66%

Salary string for zero-midpoint rows:
salary
Not disclosed    26053
0 PA                 9
Name: count, dtype: int64


In [81]:
# Stats on DISCLOSED salaries only
disclosed_mask = df_tech['salary_midpoint'] > 0
print(f"\nSalary stats (DISCLOSED rows only):")
print(df_tech[disclosed_mask]['salary_midpoint'].describe())
print(f"\nIn LPA (dividing by 100,000):")
print((df_tech[disclosed_mask]['salary_midpoint'] / 100000).describe().round(2))


Salary stats (DISCLOSED rows only):
count    6.376000e+03
mean     1.196960e+06
std      1.033215e+06
min      7.860000e+02
25%      4.000000e+05
50%      9.000000e+05
75%      1.800000e+06
max      2.500000e+07
Name: salary_midpoint, dtype: float64

In LPA (dividing by 100,000):
count    6376.00
mean       11.97
std        10.33
min         0.01
25%         4.00
50%         9.00
75%        18.00
max       250.00
Name: salary_midpoint, dtype: float64


In [82]:
# Check extreme values in disclosed salaries
disclosed_df = df_tech[df_tech['salary_midpoint'] > 0].copy()

# Convert to LPA for readable numbers
disclosed_df['salary_lpa'] = disclosed_df['salary_midpoint'] / 100000

print("Salary distribution in LPA (disclosed only):")
print(disclosed_df['salary_lpa'].describe().round(2))

# Flag suspicious outliers
very_high = disclosed_df[disclosed_df['salary_lpa'] > 100]
very_low = disclosed_df[(disclosed_df['salary_lpa'] > 0) & (disclosed_df['salary_lpa'] < 1)]

print(f"\nSuspiciously high (>100 LPA): {len(very_high)} rows")
print(very_high[['title', 'companyName', 'salary', 'salary_lpa']].head(10))

print(f"\nSuspiciously low (<1 LPA): {len(very_low)} rows")
print(very_low[['title', 'companyName', 'salary', 'salary_lpa']].head(10))

Salary distribution in LPA (disclosed only):
count    6376.00
mean       11.97
std        10.33
min         0.01
25%         4.00
50%         9.00
75%        18.00
max       250.00
Name: salary_lpa, dtype: float64

Suspiciously high (>100 LPA): 1 rows
                      title          companyName     salary  salary_lpa
1332  Senior Director AI ML  Employers4employees  2-3 Cr PA       250.0

Suspiciously low (<1 LPA): 87 rows
                                                                 title                         companyName               salary  salary_lpa
111                                    Senior Network Support Engineer                Techno One Solutions     50,000-70,000 PA       0.600
161            Corporate Trainer / Instructor - RedHat (Not Developer)                    Koenig Solutions  50,000-1.25 Lacs PA       0.875
162            Corporate Trainer / Instructor - RedHat (Not Developer)                    Koenig Solutions  50,000-1.25 Lacs PA       0.875
456   So

In [83]:
# Add salary_lpa column to df_tech for all analysis going forward
df_tech['salary_lpa'] = df_tech['salary_midpoint'] / 100000

# Fix 1: The 9 rows with "0 PA" - set to NaN (treat as not disclosed)
df_tech.loc[df_tech['salary'] == '0 PA', 'salary_lpa'] = np.nan

# Fix 2: Rows where salary < 1 LPA and > 0 - these are monthly figures, multiply by 12
monthly_mask = (df_tech['salary_lpa'] > 0) & (df_tech['salary_lpa'] < 1)
df_tech.loc[monthly_mask, 'salary_lpa'] = df_tech.loc[monthly_mask, 'salary_lpa'] * 12

print(f"Rows corrected from monthly to annual: {monthly_mask.sum()}")

# Final disclosed salary stats after fixes
disclosed_mask = df_tech['salary_lpa'] > 0
print(f"\nFinal salary stats (LPA, disclosed only):")
print(df_tech[disclosed_mask]['salary_lpa'].describe().round(2))

# Apply same salary_lpa to df_skills
df_skills['salary_lpa'] = df_skills['salary_midpoint'] / 100000
df_skills.loc[df_skills['salary_lpa'] > 0, 'salary_lpa'] = df_skills.loc[
    df_skills['salary_lpa'] > 0, 'salary_lpa']

# Final check
print(f"\nFinal disclosed count: {disclosed_mask.sum():,}")
print(f"Final disclosure rate: {disclosed_mask.sum()/len(df_tech)*100:.2f}%")

Rows corrected from monthly to annual: 87

Final salary stats (LPA, disclosed only):
count    6376.00
mean       12.06
std        10.27
min         0.09
25%         4.25
50%         9.00
75%        18.00
max       250.00
Name: salary_lpa, dtype: float64

Final disclosed count: 6,376
Final disclosure rate: 19.66%


In [84]:
# Experience text column - 2.15% missing
df_tech['experience'] = df_tech['experience'].fillna('Not Specified')

# AggregateRating and ReviewsCount - 36% missing, flag don't impute
df_tech['has_company_rating'] = df_tech['AggregateRating'].notna()
df_tech['AggregateRating'] = df_tech['AggregateRating'].fillna(0)
df_tech['ReviewsCount'] = df_tech['ReviewsCount'].fillna(0)

print("Remaining nulls in df_tech:")
print(df_tech.isnull().sum()[df_tech.isnull().sum() > 0])

Remaining nulls in df_tech:
salary_lpa    9
dtype: int64


In [85]:
import os

# Ensure output directory exists
os.makedirs('../data/processed', exist_ok=True)

# Save 1: Full cleaned dataset (all industries, all cleaning applied)
df.to_csv('../data/processed/india_jobs_cleaned.csv', index=False)
print(f"Saved india_jobs_cleaned.csv — {len(df):,} rows")

# Save 2: Tech-only dataset (your primary analysis dataset)
df_tech.to_csv('../data/processed/tech_jobs_cleaned.csv', index=False)
print(f"Saved tech_jobs_cleaned.csv — {len(df_tech):,} rows")

# Save 3: Long-format skills table (your skill analysis dataset)
df_skills.to_csv('../data/processed/skills_exploded.csv', index=False)
print(f"Saved skills_exploded.csv — {len(df_skills):,} rows")

# Save 4: Summary stats for quick Power BI reference
summary = {
    'total_raw_records': 97929,
    'total_cleaned_records': len(df),
    'total_tech_records': len(df_tech),
    'total_skill_mentions': len(df_skills),
    'unique_skills': df_skills['skill'].nunique(),
    'unique_cities': df_tech['primary_city'].nunique(),
    'unique_companies': df_tech['companyName'].nunique(),
    'salary_disclosure_rate_pct': round(disclosed_mask.sum()/len(df_tech)*100, 2),
    'role_categories': df_tech['role_category'].nunique()
}

import json
with open('../data/processed/project_summary.json', 'w') as f:
    json.dump(summary, f, indent=4)

print(f"\nSaved project_summary.json")
print(f"\nProject Summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")

Saved india_jobs_cleaned.csv — 97,105 rows
Saved tech_jobs_cleaned.csv — 32,438 rows
Saved skills_exploded.csv — 241,109 rows

Saved project_summary.json

Project Summary:
  total_raw_records: 97929
  total_cleaned_records: 97105
  total_tech_records: 32438
  total_skill_mentions: 241109
  unique_skills: 18702
  unique_cities: 428
  unique_companies: 6482
  salary_disclosure_rate_pct: 19.66
  role_categories: 19


In [87]:
# In your notebook — re-save skills_exploded with proper quoting
import pandas as pd

df_skills = pd.read_csv('../data/processed/skills_exploded.csv')

# Re-save with explicit quoting to handle any commas in skill names
df_skills.to_csv(
    '../data/processed/skills_exploded.csv',
    index=False,
    quoting=1  # csv.QUOTE_ALL — wraps every field in quotes
)

print(f"Re-saved: {df_skills.shape}")
print(f"Sample skills: {df_skills['skill'].head(5).tolist()}")

Re-saved: (241109, 15)
Sample skills: ['M.S Office', 'SAP', 'Computer Skills', 'Ms Office', 'Technical']


In [88]:
# This must exist and run BEFORE the save step
df_tech['fresher_eligible'] = df_tech['minimumExperience'] <= 2

# Then this save cell must run AFTER
df_tech.to_csv('../data/processed/tech_jobs_cleaned.csv', index=False)
print(f"✅ Saved with fresher_eligible column: {df_tech.shape}")
print(f"Columns: {df_tech.columns.tolist()}")

✅ Saved with fresher_eligible column: (32438, 28)
Columns: ['title', 'jobId', 'currency', 'jobUploaded', 'companyName', 'tagsAndSkills', 'experience', 'salary', 'location', 'companyId', 'ReviewsCount', 'AggregateRating', 'jobDescription', 'minimumSalary', 'maximumSalary', 'minimumExperience', 'maximumExperience', 'work_mode', 'primary_city', 'city_tier_group', 'metro_region', 'is_tech_role', 'role_category', 'skills_list', 'salary_midpoint', 'salary_lpa', 'has_company_rating', 'fresher_eligible']


In [89]:
import pandas as pd
df_check = pd.read_csv('../data/processed/tech_jobs_cleaned.csv')
print(f"Shape: {df_check.shape}")
print(f"'fresher_eligible' exists: {'fresher_eligible' in df_check.columns}")
print(f"Fresher eligible count: {df_check['fresher_eligible'].sum():,}")

Shape: (32438, 28)
'fresher_eligible' exists: True
Fresher eligible count: 6,556
